# 05 HRV Domain And Robustness

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    _run_setup('git fetch origin')
    current_branch = subprocess.check_output(
        'git branch --show-current',
        shell=True,
        text=True,
    ).strip()
    if current_branch != BRANCH:
        _run_setup(f'git checkout {BRANCH}')
    _run_setup(f'git pull --ff-only --autostash origin {BRANCH}')
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        _run_setup('git fetch origin')
        _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', check=False)
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## HRV Domain Sensitivity And Minimum Robustness

Outputs required by the revised plan: HRV-only result, HRV domain classifier AUC, duration/noise check, SNR 20/10/5 dB, random 3-lead dropout, V1-V6 dropout, and 500->250 Hz sampling shift.

In [ ]:
import pandas as pd

tasks = pd.read_csv('docs/revision_plan/task_board.csv')
display(tasks[tasks['id'].isin(['A6', 'B2'])])

for path in [
    'reports/revision/metrics/hrv_domain_summary.csv',
    'reports/revision/metrics/robustness_summary.csv',
    'reports/revision/tables/table_robustness.csv',
]:
    print(path, 'exists=', Path(path).exists())


## Placeholder Commands

In [ ]:
RUN_HEAVY = False

planned = {
    'HRV domain classifier': 'python scripts/revision/TBD_hrv_domain_classifier.py',
    'Robustness stress tests': 'python scripts/revision/TBD_robustness_stress.py',
}

for name, command in planned.items():
    if RUN_HEAVY:
        run(command, check=False)
    else:
        print(f'{name}: planned command -> {command}')
